# Trace Count v4: v2-Style Steering

This notebook follows `pipeline_v4_steering_codex_prompt.md`. It intentionally reuses the v2 symbolic counting setup and v2-style HuggingFace GPT-2 architecture with learned absolute positional embeddings. The new part is mechanistic: hidden-state cache, probes, direction extraction, steering, and patching.

In [1]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
if Path('/content').exists():
    repo_dir = Path('/content/Synthetic_CoT_NiaH_Count')
    if repo_dir.exists():
        os.chdir(repo_dir)
        subprocess.run(['git', 'pull'], check=False)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)
        os.chdir(repo_dir)

def patch_v4_model_hook_for_new_transformers():
    path = Path('synthetic_niah_v4/model.py')
    if not path.exists():
        print('v4 model hook patch skipped; file not found:', path)
        return
    text = path.read_text(encoding='utf-8')
    old = '''                handles.append(
                    block.register_forward_pre_hook(
                        lambda _module, inputs, layer_idx=layer_idx: (
                            apply(f"resid_pre_layer_{layer_idx}", inputs[0]),
                            *inputs[1:],
                        )
                    )
                )

                def post_hook(_module, _inputs, output, layer_idx=layer_idx):
                    hidden = apply(f"resid_post_layer_{layer_idx}", output[0])
                    return (hidden, *output[1:])
'''
    new = '''                def pre_hook(_module, inputs, layer_idx=layer_idx):
                    if not inputs:
                        return inputs
                    hidden = inputs[0]
                    if isinstance(hidden, tuple):
                        if not hidden:
                            return inputs
                        patched = (apply(f"resid_pre_layer_{layer_idx}", hidden[0]), *hidden[1:])
                        return (patched, *inputs[1:])
                    return (apply(f"resid_pre_layer_{layer_idx}", hidden), *inputs[1:])

                handles.append(
                    block.register_forward_pre_hook(pre_hook)
                )

                def post_hook(_module, _inputs, output, layer_idx=layer_idx):
                    if isinstance(output, tuple):
                        hidden = apply(f"resid_post_layer_{layer_idx}", output[0])
                        return (hidden, *output[1:])
                    return apply(f"resid_post_layer_{layer_idx}", output)
'''
    if old in text:
        path.write_text(text.replace(old, new), encoding='utf-8')
        print('Patched synthetic_niah_v4/model.py for newer Transformers/PyTorch hook outputs.')
    elif 'return apply(f"resid_post_layer_{layer_idx}", output)' in text:
        print('v4 model hook patch already present.')
    else:
        print('v4 model hook patch not applied; source pattern did not match.')

patch_v4_model_hook_for_new_transformers()

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('cwd =', Path.cwd())

cwd = /content/Synthetic_CoT_NiaH_Count


## Runtime settings

- `debug`: quick end-to-end sanity check.
- `main`: v2-style main setting: two models, one seed by default, `1..10` counts, GPT-2 learned absolute positions.
- `stage='all'` runs training, behavior eval, hidden cache, probes, direction extraction, patching, steering, plots, and a markdown/html artifact report.

In [2]:
PRESET = 'debug'  # 'debug' or 'main'
STAGE = 'all'
SEEDS = '1234'
OUT_ROOT = 'runs/synthetic_niah_v4'
RUN_NAME = ''
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
SKIP_COMPLETED = True

print({
    'PRESET': PRESET,
    'STAGE': STAGE,
    'SEEDS': SEEDS,
    'OUT_ROOT': OUT_ROOT,
    'DEVICE': DEVICE,
    'SKIP_COMPLETED': SKIP_COMPLETED,
})

{'PRESET': 'debug', 'STAGE': 'all', 'SEEDS': '1234', 'OUT_ROOT': 'runs/synthetic_niah_v4', 'DEVICE': 'cuda', 'SKIP_COMPLETED': True}


In [3]:
cmd = [
    sys.executable, '-u', '-m', 'synthetic_niah_v4.run_v4',
    '--preset', PRESET,
    '--stage', STAGE,
    '--device', DEVICE,
    '--out-root', OUT_ROOT,
    '--seeds', SEEDS,
]
if RUN_NAME:
    cmd += ['--run-name', RUN_NAME]
if SKIP_COMPLETED:
    cmd.append('--skip-completed')
print(' '.join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
captured = []
for line in proc.stdout:
    print(line, end='')
    captured.append(line.rstrip())
returncode = proc.wait()
if returncode:
    print('---- Last 120 log lines ----')
    print('\n'.join(captured[-120:]))
    raise subprocess.CalledProcessError(returncode, cmd)

RUN_DIR = None
for line in reversed(captured):
    if line.startswith('FINAL_RUN_DIR '):
        RUN_DIR = Path(line.split(' ', 1)[1].strip())
        break
if RUN_DIR is None:
    RUN_DIR = Path(OUT_ROOT) / RUN_NAME if RUN_NAME else Path(OUT_ROOT)
print('RUN_DIR =', RUN_DIR)

/usr/bin/python3 -u -m synthetic_niah_v4.run_v4 --preset debug --stage all --device cuda --out-root runs/synthetic_niah_v4 --seeds 1234 --skip-completed
[v4] stage=train
[v4 train] non_thinking seed=1234

v4 non_thinking seed=1234:   0%|          | 0/8 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/Synthetic_CoT_NiaH_Count/synthetic_niah_v4/run_v4.py", line 152, in <module>
    main()
  File "/content/Synthetic_CoT_NiaH_Count/synthetic_niah_v4/run_v4.py", line 148, in main
    run(build_parser().parse_args())
  File "/content/Synthetic_CoT_NiaH_Count/synthetic_niah_v4/run_v4.py", line 123, in run
    run_stage(stage, cfg, run_dir, vocab, args.skip_completed)
  File "/content/Synthetic_CoT_NiaH_Count/synthetic_niah_v4/run_v4.py", line 41, in run_stage
    return train_all(cfg, vocab, run_dir, skip_completed=skip_completed)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^

CalledProcessError: Command '['/usr/bin/python3', '-u', '-m', 'synthetic_niah_v4.run_v4', '--preset', 'debug', '--stage', 'all', '--device', 'cuda', '--out-root', 'runs/synthetic_niah_v4', '--seeds', '1234', '--skip-completed']' returned non-zero exit status 1.

## Result tables

The most important tables are behavior accuracy, probe results, direction diagnostics, and steering/patching. Probe accuracy alone means decodability; steering or patching is the causal check.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

tables = RUN_DIR / 'tables'
for name in [
    'behavior_accuracy_by_count.csv',
    'probe_results.csv',
    'direction_metrics.csv',
    'steering_results.csv',
    'interchange_patching_results.csv',
]:
    path = tables / name
    display(Markdown(f'### `{name}`'))
    if path.exists() and path.stat().st_size > 0:
        display(pd.read_csv(path).head(20))
    else:
        display(Markdown('_No rows._'))

## Figures

These figures summarize: probe accuracy/R2 by anchor and layer, direction agreement with the count-token unembedding, steering dose response, steering controls, and patching from donor to receiver counts.

In [ ]:
from IPython.display import Image

for fig in sorted((RUN_DIR / 'figures').glob('*.png')):
    display(Markdown(f'### `{fig.name}`'))
    display(Image(filename=str(fig), width=900))

## Save to Google Drive

In [ ]:
SAVE_TO_DRIVE = False
DRIVE_SAVE_COMPLETED = False
DRIVE_DIR = '/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/Synthetic_CoT_NiaH_Count/colab_results/'

if SAVE_TO_DRIVE and Path('/content').exists():
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    dest = Path(DRIVE_DIR) / RUN_DIR.name
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(RUN_DIR, dest)
    DRIVE_SAVE_COMPLETED = True
    print('saved to', dest)
else:
    print('SAVE_TO_DRIVE is False; local RUN_DIR:', RUN_DIR)

## Auto-disconnect Colab Runtime

This cell runs immediately after the Google Drive save cell. It only disconnects when a Drive save was confirmed; local VSCode/Jupyter runs are not force-closed by default.

In [ ]:
AUTO_DISCONNECT_AFTER_DRIVE_SAVE = True
FORCE_LOCAL_KERNEL_SHUTDOWN = False

if AUTO_DISCONNECT_AFTER_DRIVE_SAVE and globals().get("DRIVE_SAVE_COMPLETED", False):
    import time

    print("Google Drive save completed. Flushing Drive and disconnecting Colab runtime in 3 seconds...")
    time.sleep(3)
    try:
        from google.colab import drive, runtime

        try:
            drive.flush_and_unmount()
            print("Google Drive flushed and unmounted.")
        except Exception as e:
            print(f"Drive flush/unmount skipped or failed: {e}")
        runtime.unassign()
    except Exception as e:
        print(f"Colab runtime disconnect unavailable or failed: {e}")
        if FORCE_LOCAL_KERNEL_SHUTDOWN:
            import IPython

            IPython.Application.instance().kernel.do_shutdown(restart=False)
        else:
            print("Not forcing local kernel shutdown.")
else:
    print("Auto-disconnect skipped: no confirmed Google Drive save, or AUTO_DISCONNECT_AFTER_DRIVE_SAVE is False.")

## Optional GitHub result upload

In [ ]:
PUSH_RESULTS_TO_GITHUB = False
GIT_BRANCH = 'v4-results'

if PUSH_RESULTS_TO_GITHUB:
    subprocess.run(['git', 'checkout', '-B', GIT_BRANCH], check=True)
    subprocess.run(['git', 'add', str(RUN_DIR)], check=True)
    subprocess.run(['git', 'commit', '-m', f'Add v4 results {RUN_DIR.name}'], check=False)
    subprocess.run(['git', 'push', '-u', 'origin', GIT_BRANCH], check=True)
else:
    print('PUSH_RESULTS_TO_GITHUB is False')